# Experimentos a escala — matriz de escenarios × semillas

**Fase 5 · Tesis PR-196.** Corre las políticas sobre varios escenarios y semillas, y agrega el costo en **media ± desviación**. Así el resultado deja de ser un caso suelto y se vuelve robusto.

Usa el LLM **simulado** (gratis). El LLM real se reserva para una corrida final mínima con caché.

## Preparación y carga

In [ ]:
import os, sys
from pathlib import Path

raiz = Path.cwd()
while not (raiz / "src").exists():
    raiz = raiz.parent
os.chdir(raiz)
sys.path.insert(0, str(raiz / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from tesis.modelo_base import ModeloBase
from tesis.simulador.poblacion import cargar_poblacion, COHORTE_OOT
from tesis.senales.monitor import Referencia
from tesis.politicas import crear, ClienteSimulado, PoliticaLLM
from tesis.experimentos.matriz import correr_matriz, resumen_matriz

modelo = ModeloBase.cargar("models/modelo_base.pkl")
pool = cargar_poblacion(cohorte=COHORTE_OOT)
dev = cargar_poblacion(cohorte=None); dev = dev[dev["FECHA_CORTE"] < COHORTE_OOT]
referencia = Referencia.construir(dev, modelo)

def fabrica():
    return {"reglas": crear("reglas"), "multisenal": crear("multisenal"),
            "llm(sim)": PoliticaLLM(cliente=ClienteSimulado())}
print("listo")

## Correr la matriz

Imprime el progreso por escenario/semilla mientras corre (tarda ~3-4 min; las de `concept`/`mezcla` son las lentas por los re-fits). Guarda el resultado en `experiments/`.

*Para no re-correr: salta esta celda y usa la de abajo ("cargar resultado guardado").*

In [ ]:
escenarios = ["sano", "covariate", "concept", "mezcla"]
semillas = [13579, 24680, 11111]

df = correr_matriz(fabrica, escenarios, semillas, pool, modelo, referencia, N=42, verbose=True)
df.to_csv("experiments/fase5_matriz.csv", index=False)
df

## (Opcional) Cargar resultado guardado

Si ya corriste la matriz antes, esto lee el CSV al instante (sin re-correr).

In [ ]:
df = pd.read_csv("experiments/fase5_matriz.csv")
df

## Resultados agregados (media ± desviación)

In [ ]:
print("COSTO TOTAL")
resumen_matriz(df, "costo_total")

In [ ]:
print("DAÑO TOTAL")
display(resumen_matriz(df, "dano_total"))
print("Nº ACCIONES")
display(resumen_matriz(df, "n_acciones"))

## Gráfico: costo por política y escenario

Barras con la media y la barra de error = desviación entre semillas.

In [ ]:
orden_esc = ["sano", "covariate", "concept", "mezcla"]
pols = ["reglas", "multisenal", "llm(sim)"]
colores = {"reglas": "#C0392B", "multisenal": "#2E86DE", "llm(sim)": "#27AE60"}

media = df.groupby(["escenario", "politica"])["costo_total"].mean().unstack().reindex(orden_esc)
desv = df.groupby(["escenario", "politica"])["costo_total"].std().unstack().reindex(orden_esc).fillna(0)

x = np.arange(len(orden_esc)); w = 0.25
fig, ax = plt.subplots(figsize=(9, 5))
for i, pol in enumerate(pols):
    ax.bar(x + (i - 1) * w, media[pol], w, yerr=desv[pol], capsize=3, label=pol, color=colores[pol])
ax.set_xticks(x); ax.set_xticklabels(orden_esc)
ax.set_ylabel("costo total (media ± desv.)")
ax.set_title("Costo por política y escenario (menor es mejor)")
ax.legend(); ax.grid(axis="y", alpha=0.3)
plt.tight_layout(); plt.show()